<div style="background: linear-gradient(90deg, #10b981, #06b6d4); color: #042f2e; padding: 6px 14px; border-radius: 20px; font-weight: 800; display: inline-block; margin-bottom: 10px;">🏆 SCORE: Top-30 0.950 Kagriculture</div>
# 🌾 Kaggriculture Meta Field Guide: Routes, Markets & the Top-30 Playbook

**By Berat Erol ÇELİK (@beraterolelk)** · Sept 2026 · Entry deadline Sept 23 — read this before you submit

---

Kaggriculture is **not** a machine-learning competition. It is a **simultaneous-turn farming economy game**:
two agents per episode, 720 decision steps, shared market with price impact, hidden opponent state.

After auditing the public Top-30 record — boatlee's **V16-RC5** (60/60 wins vs the reconstructed 8C/4S core),
Kaito Fukami's **v27 Meta Reset** (one 719-action route in both seats, frozen-audit documented), and
yhay81's **Shop Router** (13 precomputed plans selected by revealed shop state) — the meta has three pillars:

1. **Fixed production routes** — the highest scores come from replay-derived schedules, not on-line search.
2. **Market adaptation on top of routes** — SELL-queue position matters more than what you sell.
3. **Deterministic packaging** — 20 KB, zero dependencies, zero runtime search.

Every claim below is sourced from those three public notebooks. Start from them, don't copy them —
the leaderboard punishes clones with mirror-match draws.


## 1. The three public pillars (attribution first)

| Notebook | Author | Core idea | Public record |
|---|---|---|---|
| [V16-RC5 Premium Market Lead](https://www.kaggle.com/code/boatlee/v16-rc5-high-score-8c-4s-premium-market-lead) | boatlee | Reconstructed 8 COW / 4 SHEEP route from 3 public replays of Nikita Lugovoy's `55440039` + one-turn market lead | 60/60 wins vs reconstructed core |
| [v27 Midgame Meta Reset](https://www.kaggle.com/code/kaitofukami/25-27-strict-future-v27-midgame-meta-reset) | Kaito Fukami | Same dominant opening as v26, different continuation; strict-future audit vs Top-30 | documented W-T-L with margins |
| [Shop Router 0909](https://www.kaggle.com/code/yhay81/shop-router-0909) | Yusuke Hayashi | 13 complete 719-turn plans; pick by first two revealed shops; DIG when weeds block | verified packaging |

**Common pattern:** all three ship a *precomputed* policy. Compute at home, execute on the server.


## 2. Economy rules that actually decide games

- **Shared market, queue position:** once a `SELL` adds inventory, later units clear at lower prices.
Boatlee's entire edge is a *one-turn* check: if the next scheduled sale of a premium good is imminent,
hold today's lot back. One decision, dozens of Elo points.
- **Livestock ramp:** V16-RC5 goes 4 SHEEP immediately, 8 COW by step 192. Milk/wool pay every step,
crops pay in waves — the mix smooths cashflow for fertilizer timing.
- **Day-0 shop reveal:** yhay81 branches plans on the first two shops. BAKERY wants EGG+WHEAT,
PIZZA wants MILK+TOMATO+WHEAT, YARN wants WOOL. Your route should too.
- **Weeds block planting:** the only universally justified on-line action is `DIG` when weeds block
a scheduled planting/construction step (that's yhay81's rule 2).


## 3. A minimal-but-legal agent skeleton

You do **not** need ML. You need a legal, deterministic executor. The smallest competitive core:


In [ ]:
import json

# Precompute your 719-step route offline (or adapt a public replay).
ROUTE = []  # list of {"farmer": [...], "hands": [[...], ...], "market": [[...], ...]}

PREMIUM = {"MELON", "MILK", "STRAWBERRY", "WOOL"}

def agent(obs):
    step = obs["step"]
    if step >= len(ROUTE):
        base = {"farmer": ["PASS"], "hands": [], "market": []}
    else:
        base = ROUTE[step]

    # one-turn market lead (boatlee's rule, simplified):
    # if the next route step sells the same premium good, defer today's sale.
    if step + 1 < len(ROUTE):
        nxt = ROUTE[step + 1].get("market", [])
        nxt_goods = {o[0] for o in nxt if o}
        base["market"] = [
            o for o in base.get("market", [])
            if not (o and o[0] in PREMIUM and o[0] in nxt_goods)
        ]
    return base


## 4. How to reconstruct a route from public replays (boatlee's method)

1. Pick a high-scoring public agent's **episodes** (Kaggle exposes episode replays for simulation comps).
2. For each step, take the **majority** `farmer`/`hands`/`market` action across 3 traces.
3. Normalize action formatting to the current env schema.
4. **Validate**: run your reconstruction against the source agent locally — boatlee required 60/60 wins
before trusting it.
5. Package: single `main.py`, zero imports beyond stdlib, deterministic.

**Attribution is mandatory:** Kaito's notebook ships a machine-readable provenance card (which episodes,
which snapshots, what was modified). Do the same — the community checks, and clones without credit
get called out in the forums (which kills your vote count).


## 5. The leaderboard math (why bronze is reachable)

- ~9,500 teams → 🥉 bronze = **top 957**, silver = 478, gold = 29.
- Simulation comps have **high variance**: mirror-match draws between clones, price-path luck,
and matchup sampling mean the public LB moves hundreds of places overnight.
- The proven ladder: submit a solid route → earn matches → adapt market timing → climb.

**Ethics note:** resubmitting a public agent verbatim is allowed but capped near its known ceiling —
and it farms draws against itself. Fusion (your microstructure logic + their market lead),
with documented attribution, is both more ethical and higher-EV.


## 6. Submission packaging checklist

```text
✓ main.py at archive root, defines agent(obs) -> dict
✓ zero third-party imports (json, zlib, base64 ok)
✓ 719 actions total, deterministic
✓ validate locally with kaggle_environments.make('kaggriculture', debug=True)
✓ tar.gz the file (not zip), test the roundtrip
```

## 7. Timeline

| Date | Action |
|---|---|
| Sep 19-21 | Fork a public pillar, validate locally |
| **Sep 23** | ⚠️ **Entry deadline — accept rules & submit once** |
| Sep 24-29 | Iterate market timing only |
| Sep 30 | Final deadline; select your best 2 (simulation comps usually auto-select best) |

*Sources: boatlee V16-RC5, Kaito Fukami v27, yhay81 Shop Router 0909 — all linked above with author attribution.
If you found this useful, check those three notebooks first — they are the real work.*
